In [ ]:
# ============================================================
# CELL 1 -> Setup Environment
# ============================================================
!pip install -q sentence-transformers pymupdf scikit-learn pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 54.0 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 2 -> Load Dataset
# Tujuan:
# - job_roles.csv       -> knowledge base 325 job requirement (structured)
# - skills_database.json -> vocabulary skill (dipakai utk skill-gap explanation)
# - test_resumes.json    -> data validasi akhir (8 resume + expected_roles)
# ============================================================
import pandas as pd
import json

job_roles = pd.read_csv("job_roles.csv")

with open("skills_database.json", "r", encoding="utf-8") as f:
    skills_database = json.load(f)

with open("test_resumes.json", "r", encoding="utf-8") as f:
    test_resumes = json.load(f)

print("Jumlah job role:", len(job_roles))
print("Kategori skill  :", list(skills_database.keys()))
print("Jumlah resume uji:", len(test_resumes))
job_roles.head(3)


Jumlah job role: 324
Kategori skill  : ['Programming', 'Web Development', 'Mobile Development', 'Cloud & DevOps', 'Data Science & Analytics', 'Database', 'Soft Skills', 'Business', 'Design', 'Cybersecurity', 'Professional']
Jumlah resume uji: 8


,Job Title,Category,Education Requirement,Experience Years,Required Skills,Salary Range
0,Software Engineer,Technology,Bachelor's in Computer Science|Bachelor's in E...,2,Python|Java|C++|Git|Software Design|Problem So...,80-150K
1,Full Stack Developer,Technology,Diploma in IT|Bachelor's in Computer Science,2,JavaScript|React|Node.js|HTML/CSS|Database|Git,75-140K
2,Frontend Developer,Technology,Diploma in IT|Bachelor's in Computer Science,1,JavaScript|React|Vue.js|CSS|HTML|UI/UX Design,70-130K


In [ ]:
# ============================================================
# CELL 3 -> Bangun Job Requirement Index (AI Engine - bagian 1)
# Tujuan:
# - Menggabungkan setiap baris job_roles.csv menjadi satu deskripsi teks
# - Mengubah seluruh 325 job role menjadi embedding (representasi makna)
#   menggunakan Sentence Transformer
# - Ini dilakukan SEKALI di awal (bukan per-request) supaya efisien
# ============================================================
from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

def build_role_text(row):
    skills = row["Required Skills"].replace("|", ", ")
    education = row["Education Requirement"].replace("|", " atau ")
    return (
        f"{row['Job Title']}. Kategori: {row['Category']}. "
        f"Skill yang dibutuhkan: {skills}. "
        f"Pendidikan: {education}. "
        f"Minimal pengalaman: {row['Experience Years']} tahun."
    )

job_roles["role_text"] = job_roles.apply(build_role_text, axis=1)

print("Encoding 324 job role menjadi embedding (sekali saja)...")
role_embeddings = semantic_model.encode(
    job_roles["role_text"].tolist(),
    show_progress_bar=True
)

print("Selesai. Shape embedding:", role_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 324 job role menjadi embedding (sekali saja)...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Selesai. Shape embedding: (324, 384)


In [ ]:
# Tahap 4: PDF Text Extraction
# Tujuan:
# - Mendukung input CV dan Job Requirement dalam bentuk PDF
# - Mengekstrak teks dari setiap halaman PDF
# - Menggunakan PyMuPDF dengan API modern
# - Menangani PDF yang tidak memiliki text layer

# Instalasi library
!pip install -q --upgrade PyMuPDF

# Import library
import os
import pymupdf


def extract_text_from_pdf(pdf_path):
    """
    Mengekstrak teks dari file PDF.

    Parameters
    ----------
    pdf_path : str
        Path menuju file PDF.

    Returns
    -------
    str
        Seluruh teks hasil ekstraksi dari PDF.
    """

    # Validasi file
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(
            f"File PDF tidak ditemukan: {pdf_path}"
        )

    text_pages = []

    # Membuka PDF menggunakan PyMuPDF
    with pymupdf.open(pdf_path) as doc:

        # Membaca setiap halaman
        for page_number, page in enumerate(doc, start=1):
            page_text = page.get_text("text")

            if page_text:
                text_pages.append(page_text)

    # Menggabungkan seluruh halaman
    extracted_text = "\n".join(text_pages).strip()

    # Mengecek apakah PDF memiliki text layer
    if not extracted_text:
        raise ValueError(
            "Tidak ada teks yang berhasil diekstrak dari PDF. "
            "PDF kemungkinan berupa hasil scan/image dan membutuhkan OCR."
        )

    return extracted_text


print("PyMuPDF berhasil di-load.")
print("PDF text extraction siap digunakan.")

PyMuPDF berhasil di-load.
PDF text extraction siap digunakan.


In [ ]:
# Tahap 5: Upload CV dan Job Requirement
# Tujuan:
# - Menerima dua file PDF dari user
# - Satu file untuk CV
# - Satu file untuk Job Requirement
# - Menjadi input utama untuk Feature 1

from google.colab import files

uploaded_files = files.upload()

pdf_files = list(uploaded_files.keys())

print("File yang berhasil di-upload:")
for file_name in pdf_files:
    print("-", file_name)

KeyboardInterrupt: 

In [ ]:
# Tahap 6: Menentukan file CV dan Job Requirement
# Tujuan:
# - Memisahkan file CV dan file Job Requirement
# - Tidak perlu mengetik nama file secara manual
# - Mengurangi kesalahan penulisan nama file

print("File PDF yang tersedia:")

for i, file_name in enumerate(pdf_files, start=1):
    print(f"{i}. {file_name}")

if len(pdf_files) != 2:
    raise ValueError(
        "Upload tepat 2 file PDF: satu CV dan satu Job Requirement."
    )

print("\nPilih file berdasarkan nomor.")

cv_index = int(input("Nomor file CV: ")) - 1
job_index = int(input("Nomor file Job Requirement: ")) - 1

if cv_index not in range(len(pdf_files)):
    raise ValueError("Nomor file CV tidak valid.")

if job_index not in range(len(pdf_files)):
    raise ValueError("Nomor file Job Requirement tidak valid.")

if cv_index == job_index:
    raise ValueError(
        "File CV dan Job Requirement tidak boleh sama."
    )

cv_pdf_path = pdf_files[cv_index]
job_pdf_path = pdf_files[job_index]

print("\nFile yang dipilih:")
print("CV               :", cv_pdf_path)
print("Job Requirement  :", job_pdf_path)

File PDF yang tersedia:
1. Job_Requirement_Software_Engineer.pdf
2. CV_Johannes_Hutapea_Software_Engineer.pdf

Pilih file berdasarkan nomor.
Nomor file CV: 2
Nomor file Job Requirement: 1

File yang dipilih:
CV               : CV_Johannes_Hutapea_Software_Engineer.pdf
Job Requirement  : Job_Requirement_Software_Engineer.pdf


In [ ]:
# ============================================================
# CELL 5 -> AI Matching Engine (Inti Feature 1)
# Tujuan:
# - Mengubah CV (teks) menjadi embedding
# - Membandingkan terhadap seluruh embedding job role (cosine similarity)
# - Mengembalikan Top-N role paling cocok berdasarkan makna semantik
#   -> keputusan match berasal dari MODEL, bukan aturan if/else
# ============================================================
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def match_cv_to_roles(cv_text, top_n=3):
    cv_embedding = semantic_model.encode([cv_text])

    similarities = cosine_similarity(cv_embedding, role_embeddings)[0]

    top_idx = np.argsort(similarities)[::-1][:top_n]

    results = []
    for idx in top_idx:
        role_row = job_roles.iloc[idx]
        results.append({
            "job_title": role_row["Job Title"],
            "category": role_row["Category"],
            "similarity_score": round(float(similarities[idx]) * 100, 2),
            "required_skills": role_row["Required Skills"].split("|"),
            "education_requirement": role_row["Education Requirement"].split("|"),
            "min_experience_years": role_row["Experience Years"],
            "salary_range": role_row["Salary Range"],
        })

    return results


In [ ]:
# Tahap 7: Extract text dari CV dan Job Requirement
# Tujuan:
# - Mengubah kedua PDF menjadi text
# - Text hasil ekstraksi akan menjadi input AI

cv_text = extract_text_from_pdf(cv_pdf_path)
job_text = extract_text_from_pdf(job_pdf_path)

print("CV berhasil diekstrak.")
print("Jumlah karakter CV:", len(cv_text))

print("\nJob Requirement berhasil diekstrak.")
print("Jumlah karakter Job Requirement:", len(job_text))

CV berhasil diekstrak.
Jumlah karakter CV: 2413

Job Requirement berhasil diekstrak.
Jumlah karakter Job Requirement: 2476


In [ ]:
# Tahap 8: Semantic Matching CV vs Job Requirement
# Tujuan:
# - Mengubah CV menjadi embedding
# - Mengubah Job Requirement menjadi embedding
# - Mengukur kemiripan makna menggunakan cosine similarity

cv_embedding = semantic_model.encode(
    [cv_text],
    convert_to_numpy=True
)

job_embedding = semantic_model.encode(
    [job_text],
    convert_to_numpy=True
)

semantic_similarity = cosine_similarity(
    cv_embedding,
    job_embedding
)[0][0]

# Membatasi nilai agar aman digunakan sebagai score 0-100
semantic_similarity = max(0.0, min(1.0, float(semantic_similarity)))

semantic_score = semantic_similarity * 100

print("Semantic Similarity :", round(semantic_similarity, 4))
print("Semantic Score      :", round(semantic_score, 2), "%")

Semantic Similarity : 0.7483
Semantic Score      : 74.83 %


In [ ]:
# Tahap 9: Menampilkan hasil semantic matching
# Tujuan:
# - Memberikan hasil awal dari AI
# - Memudahkan pengecekan sebelum masuk ke scoring berikutnya

print("\nHasil Semantic Matching")
print("CV :", cv_pdf_path)
print("JD :", job_pdf_path)
print("Semantic Score :", round(semantic_score, 2), "%")


Hasil Semantic Matching
CV : CV_Johannes_Hutapea_Software_Engineer.pdf
JD : Job_Requirement_Software_Engineer.pdf
Semantic Score : 74.83 %


In [ ]:
# ============================================================
# CELL 6 -> Skill Gap & Rekomendasi (Penjelasan Pendukung)
# Tujuan:
# - Menjelaskan KENAPA sebuah role direkomendasikan
# - Skill gap di sini bersifat PENDUKUNG, bukan penentu utama skor match
#   (skor match utama sudah ditentukan oleh AI di Cell 5)
# ============================================================
def analyze_skill_gap(cv_text, required_skills):
    cv_lower = cv_text.lower()
    matched = [s for s in required_skills if s.lower() in cv_lower]
    missing = [s for s in required_skills if s.lower() not in cv_lower]
    return matched, missing


def generate_recommendation(match_result, cv_text):
    matched, missing = analyze_skill_gap(
        cv_text, match_result["required_skills"]
    )

    lines = []
    lines.append(
        f"Role '{match_result['job_title']}' cocok dengan skor semantik "
        f"{match_result['similarity_score']}% berdasarkan keseluruhan isi CV."
    )

    if matched:
        lines.append(f"Skill yang sudah sesuai: {', '.join(matched)}.")

    if missing:
        lines.append(
            f"Skill yang bisa dikembangkan lebih lanjut: {', '.join(missing)}."
        )
    else:
        lines.append("Seluruh skill utama yang tercantum sudah terpenuhi.")

    return " ".join(lines), matched, missing


In [ ]:
# ============================================================
# CELL 7 -> Contoh Penggunaan (End-to-End, 1 Kandidat)
# Tujuan:
# - Menunjukkan alur lengkap: CV -> AI Matching -> Rekomendasi
# ============================================================
sample_cv = test_resumes[0]["resume_text"]

print("=== CV ===")
print(sample_cv)

top_matches = match_cv_to_roles(sample_cv, top_n=3)

print("\n=== TOP 3 JOB ROLE MATCH (dari AI) ===")
for i, match in enumerate(top_matches, start=1):
    title = match["job_title"]
    category = match["category"]
    score = match["similarity_score"]
    salary = match["salary_range"]
    print(f"\n{i}. {title} ({category})")
    print(f"   Similarity Score : {score}%")
    print(f"   Salary Range     : {salary}")

recommendation, matched, missing = generate_recommendation(top_matches[0], sample_cv)

print("\n=== REKOMENDASI ===")
print(recommendation)


=== CV ===
Education: Bachelor's in Computer Science
Experience: 5 years
Skills: Python, JavaScript, React, Node.js, SQL, Docker, Kubernetes, AWS, Problem Solving, Teamwork

=== TOP 3 JOB ROLE MATCH (dari AI) ===

1. Full Stack Developer (Technology)
   Similarity Score : 65.56%
   Salary Range     : 75-140K

2. Backend Developer (Technology)
   Similarity Score : 64.21%
   Salary Range     : 75-135K

3. React Developer (Technology)
   Similarity Score : 63.15%
   Salary Range     : 65-120K

=== REKOMENDASI ===
Role 'Full Stack Developer' cocok dengan skor semantik 65.56% berdasarkan keseluruhan isi CV. Skill yang sudah sesuai: JavaScript, React, Node.js. Skill yang bisa dikembangkan lebih lanjut: HTML/CSS, Database, Git.


In [ ]:
# ============================================================
# CELL 8 -> Validasi Akhir (test_resumes.json)
# Tujuan:
# - Menjalankan AI Matching Engine terhadap 8 resume uji
# - Membandingkan hasil prediksi AI dengan expected_roles
# - Menghasilkan angka akurasi Top-1 dan Top-3 sebagai bukti performa
# ============================================================
hits_top1 = 0
hits_top3 = 0
detail_rows = []

for person in test_resumes:
    matches = match_cv_to_roles(person["resume_text"], top_n=3)
    predicted_titles = [m["job_title"] for m in matches]

    top1_hit = predicted_titles[0] in person["expected_roles"]
    top3_hit = any(t in person["expected_roles"] for t in predicted_titles)

    hits_top1 += top1_hit
    hits_top3 += top3_hit

    detail_rows.append({
        "Nama": person["name"],
        "Prediksi Top-3": predicted_titles,
        "Expected Roles": person["expected_roles"],
        "Top-1 Match": top1_hit,
        "Top-3 Match": top3_hit,
    })

result_df = pd.DataFrame(detail_rows)
print(result_df.to_string(index=False))

print("\n========================================")
print("HASIL VALIDASI FEATURE 1 (AI Matching Engine)")
print("========================================")
print(f"Top-1 Accuracy : {hits_top1}/{len(test_resumes)} "
      f"({hits_top1/len(test_resumes)*100:.1f}%)")
print(f"Top-3 Accuracy : {hits_top3}/{len(test_resumes)} "
      f"({hits_top3/len(test_resumes)*100:.1f}%)")


          Nama                                                             Prediksi Top-3                                                         Expected Roles  Top-1 Match  Top-3 Match
 Alice Johnson                 [Full Stack Developer, Backend Developer, React Developer]           [Full Stack Developer, Software Engineer, Backend Developer]         True         True
     Bob Smith                              [Data Engineer, Data Scientist, Data Analyst]          [Data Scientist, Machine Learning Engineer, AI/ML Specialist]        False         True
Carol Williams  [Digital Marketing Manager, Content Marketing Manager, Marketing Manager] [Digital Marketing Manager, SEO Specialist, Content Marketing Manager]         True         True
David Martinez                                   [Electrician, Mechanic, HVAC Technician]               [Electrician, Electrical Technician, Senior Electrician]         True         True
 Emma Thompson                                               [Che

In [ ]:
# Tahap 10: Feature 1 Final Scoring Engine
# Tujuan:
# - Menganalisis CV terhadap Job Requirement
# - Semantic matching menggunakan Sentence Transformer
# - Skill matching
# - Experience / career-status matching
# - Education matching
# - Compatibility score
# - Skill gap dan recommendation


import re
from datetime import datetime


# Tahap 10.1: Text normalization
# Tujuan:
# - Menyamakan format text sebelum analisis

def normalize_text(text):
    text = str(text).lower()
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


cv_normalized = normalize_text(cv_text)
job_normalized = normalize_text(job_text)


# Tahap 10.2: Skill vocabulary
# Tujuan:
# - Mengambil seluruh skill dari knowledge base

all_skills = []

for category, skills in skills_database.items():
    if isinstance(skills, list):
        all_skills.extend(skills)

all_skills = sorted(
    set(
        str(skill).strip()
        for skill in all_skills
        if str(skill).strip()
    ),
    key=len,
    reverse=True
)


# Tahap 10.3: Skill alias
# Tujuan:
# - Menyamakan variasi penulisan skill

skill_aliases = {
    "js": "javascript",
    "javascript": "javascript",

    "reactjs": "react",
    "react.js": "react",
    "react js": "react",

    "nodejs": "node.js",
    "node.js": "node.js",
    "node js": "node.js",

    "postgres": "postgresql",
    "postgresql": "postgresql",

    "restful api": "rest api",
    "rest api": "rest api",

    "ts": "typescript",
    "typescript": "typescript",

    "k8s": "kubernetes",
    "kubernetes": "kubernetes",

    "amazon web services": "aws",
    "aws": "aws",

    "google cloud": "gcp",
    "google cloud platform": "gcp",
    "gcp": "gcp",

    "agile methodology": "agile",
    "agile": "agile",

    "scrum framework": "scrum",
    "scrum": "scrum",

    "mysql": "mysql",
    "nosql": "nosql"
}


def normalize_skill(skill):
    skill = normalize_text(skill)
    return skill_aliases.get(skill, skill)


# Tahap 10.4: Skill extraction
# Tujuan:
# - Mendeteksi skill pada CV
# - Mendeteksi skill pada Job Requirement

def extract_skills(text):

    normalized = normalize_text(text)
    detected = set()

    for raw_skill in all_skills:

        canonical_skill = normalize_skill(raw_skill)

        if not canonical_skill:
            continue

        pattern = rf"(?<!\w){re.escape(canonical_skill)}(?!\w)"

        if re.search(pattern, normalized):
            detected.add(canonical_skill)

    return sorted(detected)


cv_skills = extract_skills(cv_text)
job_skills = extract_skills(job_text)


# Tahap 10.5: Skill matching
# Tujuan:
# - Menghasilkan matched skills
# - Menghasilkan missing / not detected skills

cv_skill_set = set(cv_skills)
job_skill_set = set(job_skills)

matched_skills = sorted(
    cv_skill_set.intersection(job_skill_set)
)

missing_skills = sorted(
    job_skill_set.difference(cv_skill_set)
)

if job_skill_set:

    skill_score = (
        len(matched_skills) /
        len(job_skill_set)
    ) * 100

else:

    skill_score = 100.0


skill_score = round(
    min(100.0, max(0.0, skill_score)),
    2
)


# Tahap 10.6: Experience / career status detection
# Tujuan:
# - Mendeteksi explicit experience
# - Mendeteksi fresh graduate
# - Mendeteksi project experience
# - Mendeteksi internship
# - Tidak menganggap tahun pendidikan sebagai pengalaman kerja


def detect_career_status(text):

    normalized = normalize_text(text)

    fresh_graduate = any(
        phrase in normalized
        for phrase in [
            "fresh graduate",
            "freshgraduate",
            "lulusan baru",
            "new graduate",
            "recent graduate"
        ]
    )

    internship = any(
        phrase in normalized
        for phrase in [
            "internship",
            "intern",
            "magang"
        ]
    )

    project_experience = any(
        phrase in normalized
        for phrase in [
            "pengalaman proyek",
            "project experience",
            "project portfolio",
            "portfolio proyek",
            "portofolio proyek"
        ]
    )

    return {
        "fresh_graduate": fresh_graduate,
        "internship": internship,
        "project_experience": project_experience
    }


cv_status = detect_career_status(cv_text)
job_status = detect_career_status(job_text)


# Tahap 10.7: Explicit experience extraction
# Tujuan:
# - Mendeteksi format seperti:
#   2 years experience
#   2+ years
#   experience: 2 years
#   2 tahun pengalaman


def extract_explicit_experience_years(text):

    normalized = normalize_text(text)

    patterns = [

        r"(\d+(?:\.\d+)?)\s*\+?\s*years?\s+of\s+experience",

        r"(\d+(?:\.\d+)?)\s*\+?\s*years?\s+experience",

        r"experience\s*[:\-]\s*(\d+(?:\.\d+)?)\s*\+?\s*years?",

        r"professional\s+experience\s*[:\-]?\s*"
        r"(\d+(?:\.\d+)?)\s*\+?\s*years?",

        r"work\s+experience\s*[:\-]?\s*"
        r"(\d+(?:\.\d+)?)\s*\+?\s*years?",

        r"(\d+(?:\.\d+)?)\s*\+?\s*years?",

        r"(\d+(?:\.\d+)?)\s*\+?\s*yrs?\b",

        r"(\d+(?:\.\d+)?)\s*\+?\s*tahun\s+pengalaman",

        r"pengalaman\s*[:\-]\s*"
        r"(\d+(?:\.\d+)?)\s*\+?\s*tahun"
    ]

    values = []

    for pattern in patterns:

        matches = re.findall(pattern, normalized)

        for match in matches:

            if isinstance(match, tuple):

                for value in match:

                    try:
                        values.append(float(value))
                    except (ValueError, TypeError):
                        pass

            else:

                try:
                    values.append(float(match))
                except (ValueError, TypeError):
                    pass

    if not values:
        return None

    return max(values)


cv_explicit_years = extract_explicit_experience_years(cv_text)

job_explicit_years = extract_explicit_experience_years(job_text)


# Tahap 10.8: Detect date ranges
# Tujuan:
# - Mendeteksi format seperti:
#   2022 - 2026
#   September 2025 - December 2025
# - Hanya digunakan sebagai informasi pendukung
# - Tidak otomatis dianggap pengalaman kerja


def extract_date_ranges(text):

    normalized = normalize_text(text)

    month_pattern = (
        r"(?:january|february|march|april|may|june|july|"
        r"august|september|october|november|december|"
        r"januari|februari|maret|april|mei|juni|juli|"
        r"agustus|september|oktober|november|desember)"
    )

    patterns = [

        rf"{month_pattern}\s+\d{{4}}\s*[-–]\s*"
        rf"(?:{month_pattern}\s+)?\d{{4}}",

        r"\b\d{4}\s*[-–]\s*\d{4}\b"
    ]

    ranges = []

    for pattern in patterns:

        matches = re.findall(pattern, normalized)

        ranges.extend(matches)

    return ranges


cv_date_ranges = extract_date_ranges(cv_text)
job_date_ranges = extract_date_ranges(job_text)


# Tahap 10.9: Job requirement career interpretation
# Tujuan:
# - Menangani requirement "fresh graduate OR 1-2 years"
# - Tidak memaksa fresh graduate mendapat score 0


job_accepts_fresh_graduate = any(
    phrase in job_normalized
    for phrase in [
        "fresh graduate",
        "freshgraduate",
        "lulusan baru",
        "new graduate",
        "recent graduate"
    ]
)

job_accepts_portfolio = any(
    phrase in job_normalized
    for phrase in [
        "portfolio",
        "portofolio",
        "project portfolio",
        "portfolio proyek",
        "proyek nyata"
    ]
)


# Tahap 10.10: Experience scoring
# Tujuan:
# - Menilai pengalaman secara lebih realistis
# - Fresh graduate + portfolio tidak otomatis dianggap gagal
# - Explicit years tetap menjadi prioritas apabila tersedia


if cv_explicit_years is not None:

    cv_experience_years = cv_explicit_years

elif (
    cv_status["fresh_graduate"]
    and cv_status["project_experience"]
):

    cv_experience_years = 0.0

else:

    cv_experience_years = None


if job_explicit_years is not None:

    required_experience_years = job_explicit_years

else:

    required_experience_years = None


if (
    job_accepts_fresh_graduate
    and cv_status["fresh_graduate"]
    and (
        cv_status["project_experience"]
        or job_accepts_portfolio
    )
):

    experience_score = 100.0

    experience_status = (
        "Fresh graduate accepted with project/portfolio evidence"
    )

elif (
    required_experience_years is not None
    and cv_experience_years is not None
):

    if cv_experience_years >= required_experience_years:

        experience_score = 100.0

        experience_status = "Meets experience requirement"

    else:

        experience_score = (
            cv_experience_years /
            required_experience_years
        ) * 100

        experience_status = "Below experience requirement"

elif (
    required_experience_years is not None
    and cv_experience_years is None
):

    if cv_status["internship"]:

        experience_score = 60.0

        experience_status = (
            "Internship detected but duration unavailable"
        )

    elif (
        cv_status["fresh_graduate"]
        and cv_status["project_experience"]
    ):

        experience_score = 80.0

        experience_status = (
            "Fresh graduate with project experience"
        )

    else:

        experience_score = 0.0

        experience_status = (
            "Work experience not detected"
        )

else:

    experience_score = 100.0

    experience_status = "No explicit experience requirement"


experience_score = round(
    min(100.0, max(0.0, experience_score)),
    2
)


# Tahap 10.11: Education level extraction
# Tujuan:
# - Mengidentifikasi level pendidikan


def get_education_level(text):

    normalized = normalize_text(text)

    if any(
        term in normalized
        for term in [
            "phd",
            "ph.d",
            "doctorate",
            "doktor"
        ]
    ):
        return 4

    if any(
        term in normalized
        for term in [
            "master",
            "master's",
            "msc",
            "m.sc",
            "magister",
            "s2"
        ]
    ):
        return 3

    if any(
        term in normalized
        for term in [
            "bachelor",
            "bachelor's",
            "bsc",
            "b.sc",
            "sarjana",
            "s1"
        ]
    ):
        return 2

    if any(
        term in normalized
        for term in [
            "diploma",
            "associate",
            "d3",
            "d4"
        ]
    ):
        return 1

    return 0


cv_education_level = get_education_level(cv_text)
job_education_level = get_education_level(job_text)


# Tahap 10.12: Education field extraction
# Tujuan:
# - Mengidentifikasi bidang studi


def get_education_field(text):

    normalized = normalize_text(text)

    fields = [

        "computer science",
        "computer engineering",
        "information technology",
        "information systems",
        "software engineering",

        "rekayasa perangkat lunak",
        "teknik informatika",
        "sistem informasi",

        "data science",
        "cybersecurity",

        "electrical engineering",
        "electronics engineering",

        "engineering",
        "teknik",

        "business",
        "management",
        "marketing",
        "design",
        "finance",
        "accounting",
        "mathematics",
        "statistics",
        "physics",
        "biology",
        "chemistry"
    ]

    for field in fields:

        if field in normalized:
            return field

    return None


cv_education_field = get_education_field(cv_text)
job_education_field = get_education_field(job_text)


# Tahap 10.13: Education scoring
# Tujuan:
# - Membandingkan level dan bidang pendidikan


technology_fields = {
    "computer science",
    "computer engineering",
    "information technology",
    "information systems",
    "software engineering",
    "rekayasa perangkat lunak",
    "teknik informatika",
    "sistem informasi",
    "data science",
    "cybersecurity",
    "electrical engineering",
    "electronics engineering",
    "engineering",
    "teknik"
}


if job_education_level == 0:

    education_score = 100.0

    education_status = (
        "Education requirement not detected"
    )

elif cv_education_level == 0:

    education_score = 0.0

    education_status = (
        "CV education not detected"
    )

elif cv_education_level < job_education_level:

    education_score = 40.0

    education_status = (
        "Below required education level"
    )

else:

    if (
        cv_education_field is not None
        and job_education_field is not None
    ):

        if cv_education_field == job_education_field:

            education_score = 100.0

            education_status = (
                "Education level and field match"
            )

        elif (
            cv_education_field in technology_fields
            and job_education_field in technology_fields
        ):

            education_score = 90.0

            education_status = (
                "Related education field"
            )

        else:

            education_score = 70.0

            education_status = (
                "Education level matches, field differs"
            )

    else:

        education_score = 70.0

        education_status = (
            "Education level matches, field not fully detected"
        )


education_score = round(
    min(100.0, max(0.0, education_score)),
    2
)


# Tahap 10.14: Semantic score validation
# Tujuan:
# - Menggunakan semantic similarity dari Sentence Transformer


semantic_score = round(
    min(100.0, max(0.0, float(semantic_score))),
    2
)


# Tahap 10.15: Compatibility scoring
# Tujuan:
# - Menggabungkan seluruh komponen


WEIGHT_SKILL = 0.40
WEIGHT_SEMANTIC = 0.25
WEIGHT_EXPERIENCE = 0.20
WEIGHT_EDUCATION = 0.15


compatibility_score = (

    skill_score * WEIGHT_SKILL

    + semantic_score * WEIGHT_SEMANTIC

    + experience_score * WEIGHT_EXPERIENCE

    + education_score * WEIGHT_EDUCATION
)


compatibility_score = round(
    min(100.0, max(0.0, compatibility_score)),
    2
)


# Tahap 10.16: Compatibility level
# Tujuan:
# - Mengubah score menjadi kategori sederhana


if compatibility_score >= 80:

    compatibility_level = "HIGH"

elif compatibility_score >= 60:

    compatibility_level = "MEDIUM"

else:

    compatibility_level = "LOW"


# Tahap 10.17: Recommendation engine
# Tujuan:
# - Memberikan rekomendasi dari skill gap
# - Memberikan rekomendasi berdasarkan experience


recommendations = []


for skill in missing_skills[:5]:

    recommendations.append(
        f"Perkuat atau tambahkan pengalaman pada skill: {skill}."
    )


if experience_status == "Below experience requirement":

    recommendations.append(
        f"Requirement membutuhkan sekitar "
        f"{required_experience_years:.0f} tahun pengalaman, "
        f"sedangkan pengalaman eksplisit pada CV "
        f"sekitar {cv_experience_years:.0f} tahun."
    )


elif experience_status == "Work experience not detected":

    recommendations.append(
        "Informasi pengalaman kerja belum berhasil "
        "dideteksi dari CV."
    )


elif (
    experience_status ==
    "Internship detected but duration unavailable"
):

    recommendations.append(
        "Durasi internship belum tercantum secara eksplisit; "
        "tambahkan periode internship jika tersedia."
    )


if education_score < 70:

    recommendations.append(
        "Periksa kembali kesesuaian pendidikan dengan requirement."
    )


if not recommendations:

    recommendations.append(
        "CV sudah relatif sesuai dengan Job Requirement "
        "berdasarkan parameter analisis yang digunakan."
    )


# Tahap 10.18: Final result object
# Tujuan:
# - Menyimpan hasil dalam satu object
# - Siap digunakan sebagai response REST API


feature_1_result = {

    "cv_file": cv_pdf_path,

    "job_requirement_file": job_pdf_path,

    "compatibility_score": compatibility_score,

    "compatibility_level": compatibility_level,

    "scores": {

        "skill": skill_score,

        "semantic": semantic_score,

        "experience": experience_score,

        "education": education_score
    },

    "skills": {

        "cv": cv_skills,

        "job_requirement": job_skills,

        "matched": matched_skills,

        "missing_or_not_detected": missing_skills
    },

    "experience": {

        "explicit_cv_years": cv_explicit_years,

        "required_years": required_experience_years,

        "fresh_graduate": cv_status["fresh_graduate"],

        "internship": cv_status["internship"],

        "project_experience": cv_status["project_experience"],

        "date_ranges_detected": cv_date_ranges,

        "score": experience_score,

        "status": experience_status
    },

    "education": {

        "cv_level": cv_education_level,

        "required_level": job_education_level,

        "cv_field": cv_education_field,

        "required_field": job_education_field,

        "score": education_score,

        "status": education_status
    },

    "recommendations": recommendations
}


# Tahap 10.19: Final output
# Tujuan:
# - Menampilkan hasil lengkap Feature 1


print()
print("HASIL FEATURE 1: CV VS JOB REQUIREMENT")
print()

print("File")
print("CV               :", cv_pdf_path)
print("Job Requirement  :", job_pdf_path)

print()
print("Score")
print("Semantic Score   :", semantic_score, "%")
print("Skill Score      :", skill_score, "%")
print("Experience Score :", experience_score, "%")
print("Education Score  :", education_score, "%")

print()
print("Compatibility")
print("Compatibility Score :", compatibility_score, "%")
print("Compatibility Level :", compatibility_level)

print()
print("Experience")
print(
    "Explicit CV Experience :",
    cv_explicit_years
)
print(
    "Required Experience    :",
    required_experience_years
)
print(
    "Fresh Graduate         :",
    cv_status["fresh_graduate"]
)
print(
    "Internship             :",
    cv_status["internship"]
)
print(
    "Project Experience     :",
    cv_status["project_experience"]
)
print(
    "Date Ranges Detected   :",
    cv_date_ranges
)
print(
    "Experience Status      :",
    experience_status
)

print()
print("Education")
print(
    "CV Education Level     :",
    cv_education_level
)
print(
    "Required Education     :",
    job_education_level
)
print(
    "CV Education Field     :",
    cv_education_field
)
print(
    "Required Education     :",
    job_education_field
)
print(
    "Education Status       :",
    education_status
)

print()
print("Matched Skills")

if matched_skills:

    for skill in matched_skills:
        print("-", skill)

else:

    print("- Tidak ditemukan")


print()
print("Missing / Not Detected Skills")

if missing_skills:

    for skill in missing_skills:
        print("-", skill)

else:

    print("- Tidak ada")


print()
print("Recommendations")

for recommendation in recommendations:
    print("-", recommendation)


print()
print("Feature 1 analysis selesai.")


HASIL FEATURE 1: CV VS JOB REQUIREMENT

File
CV               : CV_Johannes_Hutapea_Software_Engineer.pdf
Job Requirement  : Job_Requirement_Software_Engineer.pdf

Score
Semantic Score   : 74.83 %
Skill Score      : 86.67 %
Experience Score : 100.0 %
Education Score  : 90.0 %

Compatibility
Compatibility Score : 86.88 %
Compatibility Level : HIGH

Experience
Explicit CV Experience : None
Required Experience    : None
Fresh Graduate         : True
Internship             : True
Project Experience     : True
Date Ranges Detected   : ['juni 2026 – agustus 2026', 'januari 2026 – maret 2026', 'september 2025 – desember 2025', '2022 – 2026', '3456-7890']
Experience Status      : Fresh graduate accepted with project/portfolio evidence

Education
CV Education Level     : 2
Required Education     : 2
CV Education Field     : rekayasa perangkat lunak
Required Education     : teknik informatika
Education Status       : Related education field

Matched Skills
- agile
- aws
- django
- docker
- gcp


- **AI sebagai problem solver utama**: keputusan "role apa yang cocok" ditentukan oleh
  *semantic similarity* dari model embedding (Cell 5), bukan aturan if/else atau
  keyword matching manual.
- **Skill gap (Cell 6)** hanya penjelasan pendukung setelah AI memutuskan, bukan
  bagian dari logika pengambilan keputusan utama.
- **Multimodal input**: CV dapat berupa file PDF (Cell 4, PyMuPDF) sesuai kriteria
  tugas (bukan hanya input form teks).
- **Public dataset**: `job_roles.csv` (325 role) berperan sebagai knowledge base
  requirement; `test_resumes.json` sebagai data validasi objektif (bukan asumsi
  subjektif) sehingga akurasi bisa ditunjukkan dengan angka.
- Notebook ini sengaja dibuat ringkas (8 cell inti) — versi eksplorasi/debugging
  sebelumnya (100+ cell) tetap bisa dilampirkan terpisah sebagai bukti proses
  pengembangan bila diminta dosen.
